# 02. VLM 요청과 응답 검증 / 60분

**강사 설명 → 완성된 셀 실행 → 조건 비교 → 결과 해설** 순서입니다. 코드 작성 과제나 관찰 답안 제출은 없습니다.

이 단원은 모델 응답을 읽는 단계입니다. JSON에 FOLLOW가 있어도 아직 거북이에 속도를 보내지 않습니다.

[단원 실행 안내](README.md) · [슬라이드 실습 시작](https://docs.google.com/presentation/d/1J5Jw6Lg-2oYFKec-RROYlt6nxtG-IHVHd7hLwH8gd9s/edit#slide=id.d4_lab02_start)

각 단원은 새 커널에서 시작합니다. 노트북 셀은 위에서 아래로 실행하고, `bash` 명령은 지정한 터미널에서 실행합니다. 결과가 이미 있으면 재사용 여부와 실행 이름을 확인합니다.


## 1. 아키텍처 — 지금 실행하는 부분

![VLM 요청과 응답 검증의 입력·처리·출력](../교재그림/guided-vla/day4-02.svg)

hand_prompt는 원본 이미지만 보고 FOLLOW/STOP을 반환하도록 요청합니다. parse_hand_decision은 중복 key가 사라지기 전에 검사하고, 필드·자료형·허용값을 확인합니다. FIST와 FOLLOW의 모순은 형식상 JSON이어도 거부합니다. None은 검증 실패이며 다음 제어 단계에서 STOP으로 연결됩니다.

**그림을 읽는 순서:** 1→2→3 뒤 아래 4→5→6으로 이어집니다. 실시간 반복은 새 관측이 다음 입력으로 들어오는 과정이며, 저장 결과를 읽는 셀은 모델을 다시 실행하지 않습니다.


### 전체 실행 지도에서 현재 단원 찾기

![4일차 실행 경계 전체 지도](../교재그림/guided-vla/detailed/day4-system.svg)

터미널 A/B/C는 실행·관찰 역할이고 ROS 노드와 HTTP 서비스는 실제 처리 주체입니다. 02는 파일 요청만, 05는 독립 YOLO 실행만 다루므로 매 단원에서 모든 상자를 동시에 실행하지 않습니다. 아래 단원별 도식과 함께 현재 경로를 확인합니다.

## 상세 해설 — 앞선 수업 없이도 흐름을 읽기

아래 설명은 1~3일차에서 배운 개념도 이번 실습의 입력·코드·출력에 맞춰 다시 풉니다. 실행 중에는 필요한 부분을 읽고, 수업 뒤에는 독립적인 참고 자료로 사용할 수 있습니다. 추가 작성 과제는 없습니다.

### 1. 기초 복습: 이미지 요청과 구조화 응답

- VLM은 이미지와 텍스트를 함께 입력받습니다.
- base64는 전송 표현이며 영상 인식 결과가 아닙니다.
- HTTP 성공·JSON 문법·판단의 정확성은 별개입니다.
- schema 요청 이후에도 로컬 parser가 검사합니다.

일반 텍스트 LLM 요청의 content에는 문자열이 들어가지만 이번 요청에는 실제 이미지 bytes를 base64로 바꾼 images 필드도 있습니다. base64는 이진 데이터를 JSON에 실어 보내기 위한 표현이며 특징 추출이나 압축 모델이 아닙니다. 파일 이름이 open_palm이라고 모델에 알려 정답을 유도하지 않고 원본 픽셀을 제공합니다.

프롬프트는 한 손의 모양을 FOLLOW 또는 STOP으로 판단하도록 범위를 제한합니다. 좌표와 속도는 요청하지 않습니다. HAND_SCHEMA는 command·hand_state·frame_id·reason 네 필드의 형식을 요구합니다. 서버에 schema를 전달했다고 애플리케이션 검증이 끝나는 것은 아닙니다. HTTP가 성공해도 message.content가 잘못된 JSON일 수 있고, JSON이 올바르더라도 실제 주먹을 NON_FIST로 오판할 수 있습니다.

temperature=0은 출력 변동을 줄이는 설정입니다. 동일 모델·입력의 모든 환경에서 정답이나 완전한 재현성을 보장한다는 뜻은 아닙니다. reason은 모델이 제공한 설명 문자열이며 영상의 사실 여부를 증명하는 인증값이 아닙니다.


### 2. 실행 아키텍처: 파일에서 Ollama까지

![실행 주체와 데이터 경계](../교재그림/guided-vla/detailed/day4-02.svg)

- CLI → 이미지 파일 읽기 → base64 + hand_prompt
- /api/show에서 vision 기능 확인
- /api/chat → raw 문자열 → parse_hand_decision
- 원문·판정·단계·시간을 JSONL 한 행에 저장

scripts/live_hand_decision.py는 ROS 제어기와 별개로 동작하는 요청 프로그램입니다. argparse로 --image·--model·--url을 읽고 요청마다 frame_id를 만듭니다. model_show 단계에서 모델이 vision 기능을 지원하는지 확인한 다음 image_read 단계에서 파일을 읽습니다. 이 순서를 보면 파일 누락과 모델 선택 오류를 분리할 수 있습니다.

inference 단계에서 /api/chat에 POST하고 raw 응답을 먼저 보존합니다. parse 단계에서 검증한 뒤 정상일 때 completed로 바뀝니다. JSONL은 여러 JSON을 하나의 큰 배열로 합치는 형식이 아니라 요청 한 건을 한 줄씩 추가하는 형식입니다. 같은 frame_id의 image·raw·decision을 함께 읽어야 다른 요청의 원문과 판정을 혼동하지 않습니다.

이 경로에는 /turtle1/cmd_vel 발행이 없습니다. 완성 셀의 parser 비교도 모델을 호출하지 않고 준비된 문자열을 검사합니다. 합성 문자열 검사, 실제 이미지 모델 요청, ROS 이동 실행은 각각 다른 범위를 검증하므로 순서대로 이해합니다.


### 3. 코드 추적: 파싱과 의미 관계 검사

- object_pairs_hook으로 중복 key를 먼저 거부합니다.
- 필드는 정확히 네 개, 값은 문자열이어야 합니다.
- FOLLOW는 hand_state=NON_FIST와 함께만 허용합니다.
- 정상 dict 또는 None; 실제 frame 대조는 다음 단계

보통 json.loads로 dict를 만들면 동일 key가 두 번 나온 흔적이 사라지고 뒤 값이 남을 수 있습니다. parse_hand_decision은 object_pairs_hook으로 key/value 쌍을 검사해 command가 중복된 모호한 응답을 거부합니다. 허용하지 않은 추가 필드도 거부하므로 모델이 임의 speed를 넣어도 제어 입력으로 들어오지 않습니다.

이어서 자료형·허용 command·손 상태·비어 있지 않은 frame_id·reason 길이를 검사합니다. 문법적으로 정상인 {command:FOLLOW, hand_state:FIST} 관계는 의미가 모순되므로 None입니다. 반대로 STOP과 NON_FIST의 조합은 보수적인 정지일 수 있어 parser가 금지하지 않습니다. 파서의 승인 범위와 모델의 인식 정확도는 분리해서 읽어야 합니다.

여기서 frame_id가 비어 있지 않다는 것은 형식 검사입니다. 실제 관측과 같은 ID인지는 follow_motion이 비교합니다. parser가 dict를 반환했다는 이유만으로 새 영상의 좌표와 결합하면 다른 장면의 판단이 적용될 수 있습니다.


### 4. 결과 해석: 요청 오류와 모델 STOP

- stage와 error로 네트워크·파일·파싱 오류를 찾습니다.
- 정상 STOP은 요청 실패와 다른 결과입니다.
- latency_ms와 total_ms의 포함 범위를 구분합니다.
- 프롬프트 수정 시 parser와 실패 사례도 함께 봅니다.

고정 문자열 비교에서는 정상 FOLLOW, 정상 STOP, 중복 key, 모순된 상태의 승인 여부가 핵심입니다. 실제 이미지 요청에서는 그 위에 영상 의미가 맞는지도 확인합니다. 열린 손에 STOP이 나왔다면 형식은 통과했지만 의미적으로 보수적인 오판일 수 있습니다. timeout이나 잘못된 JSON은 모델 STOP과 같은 사건으로 집계하면 안 됩니다.

live_hand_decision의 total_ms는 준비부터 끝까지, latency_ms는 inference 시작 이후 응답을 읽고 검증을 마칠 때까지입니다. 순수 신경망 GPU 연산 시간과 동일하지 않습니다. 첫 요청의 적재·서버 상태가 지연에 영향을 줄 수 있으므로 다른 모델과 비교할 때 입력·요청 조건·준비 상태를 맞춥니다.

프롬프트에 새로운 command를 추가하려면 HAND_SCHEMA와 parser의 허용값, 이후 제어기의 처리까지 함께 설계해야 합니다. 문장만 바꾸고 검증을 느슨하게 해 실행되게 만드는 방식은 오류 위치를 숨깁니다. 현재 수업에서는 두 명령을 유지하고 원문과 거부 이유를 읽는 것으로 충분합니다.


## 2. 코드 위치와 변경 범위

| 구분 | 이 단원에서 확인할 내용 |
|---|---|
| 실제 소스 | [ros_ws/src/nepes_vla_demo/nepes_vla_core/hand_follow.py](ros_ws/src/nepes_vla_demo/nepes_vla_core/hand_follow.py) |
| 바꾸어 볼 조건 | `--image 입력 파일` |
| 예상되는 차이 | 정상 FOLLOW, 정상 STOP, 중복 key, 상태 모순을 같은 parser에 통과시켜 승인과 거부를 비교합니다. 실제 이미지 결과는 원문과 화면을 함께 읽습니다. |
| 직접 구현·수정할 때 | hand_prompt를 수정해도 parser의 허용 필드와 명령은 함께 유지해야 합니다. 속도·좌표를 모델에게 임의로 생성하도록 확장하지 않습니다. frame_id가 실제 관측과 같은지는 다음 제어 단계가 검사합니다. |

설정값 변경은 아래 준비된 비교 실행으로 진행합니다. 소스를 직접 변경하는 경우에는 실행 중인 프로세스를 종료하고 다시 실행해야 합니다. ROS 패키지 파일은 재빌드·환경 source까지 확인합니다.


## 3. 준비 — 실제 실행 코드와 같은 함수를 사용합니다

아래 CPU 예제는 카메라·모델·ROS 없이 자료의 흐름을 분리해 보여 줍니다. 통과가 실제 인식 성공이나 장비 성능을 뜻하지는 않습니다.


In [ ]:
from pathlib import Path
import sys, json, inspect, uuid
from datetime import datetime, timezone
from IPython.display import display, Markdown, Code
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'09.Pseudo-VLA_v2').is_dir())
LESSON = ROOT/'09.Pseudo-VLA_v2'
if str(LESSON) not in sys.path:
    sys.path.insert(0, str(LESSON))
# 매 단원 새 커널: 다른 일차의 common 모듈이 남아 있지 않게 합니다.
RUN = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '-' + uuid.uuid4().hex[:8]
OUT = ROOT/'Data/outputs/09.Pseudo-VLA_v2/guided'/RUN

# 각 행은 고정 입력의 한 조건입니다. 합성 예제 통과를 실제 장비의 정확도·속도로 해석하지 않습니다.
def show_rows(rows):
    """조건·결과를 한 표에서 읽습니다. 수강생이 답안 dict를 채우지 않습니다."""
    if not rows:
        print('표시할 결과 없음')
        return
    keys = list(dict.fromkeys(key for row in rows for key in row))
    def value(v):
        return str(v).replace('|', '/').replace('\n', ' ')
    lines = [' | '.join(keys), ' | '.join(['---']*len(keys))]
    lines += [' | '.join(value(row.get(key, '')) for key in keys) for row in rows]
    display(Markdown('\n'.join(lines)))

# 비교 출력을 자동 저장합니다. 수강생이 관찰 답안을 작성하거나 수치를 옮겨 적지 않습니다.
def save_demo(name, rows):
    """합성 예제임을 파일에도 명시합니다. 모델 측정값이나 수강생 답안이 아닙니다."""
    OUT.mkdir(parents=True, exist_ok=True)
    path = OUT/(name+'.json')
    path.write_text(json.dumps({'kind':'synthetic-guided-demo', 'model_measured':False,
                               'rows':rows}, ensure_ascii=False, indent=2)+'\n')
    print('자동 저장:', path)
print('실행 폴더:', LESSON)


### 완성 코드 읽기

아래 셀은 실행에 쓰는 함수 본문을 그대로 표시합니다. **입력 검사 → 조건 분기 → 결과 반환**을 순서대로 읽습니다. 노트북에 복사한 별도 정답과 실제 동작이 어긋나지 않도록 같은 소스를 조회합니다.


In [ ]:
from common.hand_follow import hand_prompt, parse_hand_decision
# 함수의 실제 파일과 주석을 함께 표시합니다. 이 셀은 소스를 수정하지 않습니다.
for function in (hand_prompt, parse_hand_decision,):
    print(inspect.getsourcefile(function))
    display(Code(inspect.getsource(function), language='python'))


### 한 줄씩 읽는 처리 과정

hand_prompt는 원본 이미지만 보고 FOLLOW/STOP을 반환하도록 요청합니다. parse_hand_decision은 중복 key가 사라지기 전에 검사하고, 필드·자료형·허용값을 확인합니다. FIST와 FOLLOW의 모순은 형식상 JSON이어도 거부합니다. None은 검증 실패이며 다음 제어 단계에서 STOP으로 연결됩니다.

hand_prompt를 수정해도 parser의 허용 필드와 명령은 함께 유지해야 합니다. 속도·좌표를 모델에게 임의로 생성하도록 확장하지 않습니다. frame_id가 실제 관측과 같은지는 다음 제어 단계가 검사합니다.


## 4. 조건을 바꾸어 실행하고 표로 비교하기

정상 FOLLOW, 정상 STOP, 중복 key, 상태 모순을 같은 parser에 통과시켜 승인과 거부를 비교합니다. 실제 이미지 결과는 원문과 화면을 함께 읽습니다.

아래 입력은 경계 조건을 설명하기 위한 고정 예제입니다. 주석을 읽고 셀 전체를 실행하면 비교표와 검사 결과가 자동으로 나옵니다.


In [ ]:
from common.hand_follow import hand_prompt, parse_hand_decision

# 실제 요청 함수가 만드는 프롬프트를 읽습니다. 모델 응답을 만드는 코드는 아닙니다.
print(hand_prompt('demo-1'))
valid = dict(command='FOLLOW', hand_state='NON_FIST', frame_id='demo-1', reason='open palm')
# 아래 응답 문자열은 parser를 설명하는 고정 예제이며 실제 VLM 측정값이 아닙니다.
cases = {
    '정상 FOLLOW': json.dumps(valid),
    '정상 STOP': json.dumps({**valid, 'command': 'STOP', 'hand_state': 'FIST'}),
    '상태 모순': json.dumps({**valid, 'hand_state': 'FIST'}),
    '추가 필드': json.dumps({**valid, 'speed': 1}),
    '중복 key': json.dumps(valid)[:-1] + ', "command": "STOP"}',
}
rows = []
for name, raw in cases.items():
    decision = parse_hand_decision(raw)  # 실패 시 예외 대신 None을 반환하는 계약입니다.
    rows.append(dict(case=name, raw=raw, accepted=decision is not None, decision=decision))
# 각 행은 고정 입력의 한 조건입니다. 합성 예제 통과를 실제 장비의 정확도·속도로 해석하지 않습니다.
show_rows(rows)
assert [r['accepted'] for r in rows] == [True, True, False, False, False]
# 비교 출력을 자동 저장합니다. 수강생이 관찰 답안을 작성하거나 수치를 옮겨 적지 않습니다.
save_demo('parser-cases', rows)


## 5. 실제 입력 실행 — 화면과 로그 연결

본수업은 로컬 카메라 화면을 함께 보며 역할을 교대합니다. 아래의 A/B/C는 터미널 역할입니다. 카메라와 제어기는 각각 하나만 실행합니다. 실제 실행의 지연과 인식 결과는 위 합성 예제와 별도로 확인합니다.


> **구분: 수업 핵심** — 강사 설명을 들은 뒤 수강생 실습 구간에서 진행합니다.

## 실제 픽셀 요청

```bash
python /workspace/09.Pseudo-VLA_v2/scripts/live_hand_decision.py --image /workspace/Data/images/07.Mediapipe/person_open_palm.png
```

직접 촬영한 주먹/비주먹/손 없음/가림 사진도 `--image`로 각각 실행합니다. 파일 이름을 정답으로 전달하지 않습니다.
VLM 원문·파싱 결과·지연이 Data에 저장됩니다. 모델 첫 로드가 느린 경우 준비 요청을 한 뒤 다시 측정합니다.
정지 이미지 시험의 180초 timeout은 로봇 제어 허용 시간과 다릅니다.

**실행 위치:** 위 명령은 JupyterLab의 Terminal(컨테이너 내부)에서 실행합니다. 기본 결과 파일은 `/workspace/Data/outputs/09.Pseudo-VLA_v2/hand-decisions.jsonl`이며 한 시도당 `image,frame_id,model,raw,decision,stage,latency_ms,total_ms`가 한 JSON 행으로 추가됩니다. `stage`는 `preparation`(URL 준비), `model_show`(모델 조회), `image_read`(이미지 읽기), `inference`(요청·응답), `parse`(JSON 판정), `completed` 중 하나입니다. 오류 시 해당 단계와 `error,error_type`을 먼저 확인합니다. `latency_ms`는 추론 요청 이후 시간이며 요청 전 실패는 `null`, `total_ms`는 준비를 포함한 전체 시간입니다. `stage=parse`의 `decision=None`은 파싱 실패이고 `decision.hand_state`가 실제 손과 다른 것은 의미 판단 실패입니다. 둘을 다른 열에 확인하세요.

주먹·손 없음·가림 조건은 본인이 촬영한 파일을 `/workspace/Data/images/09.Pseudo-VLA/`에 저장하고 같은 `--image <절대경로>` 인자로 전달합니다. 원본을 준비하지 못한 조건은 미측정으로 둡니다. 네 조건의 입력 경로·기대 상태·실제 상태·schema 통과·잘못된 FOLLOW·불필요 STOP·latency_ms를 `/workspace/Data/outputs/course-review/09.Pseudo-VLA_v2/`의 개인 기록에 저장합니다. 기존 jsonl은 추가 기록이므로 frame_id와 image를 확인해 이번 실행만 대조합니다.

**JSONL 자체가 없거나 새 행이 추가되지 않았다면:** 출력 디렉터리 권한·마운트·남은 공간을 확인하고 Terminal에 표시된 결과와 traceback을 보존합니다. 로그 저장에 실패해도 원래 요청 오류 traceback을 함께 출력합니다. 기존 실패 행은 지우지 않으며 재시도는 새 frame_id로 추가됩니다.


## 6. 결과 해설과 다음 단계

이 단원은 모델 응답을 읽는 단계입니다. JSON에 FOLLOW가 있어도 아직 거북이에 속도를 보내지 않습니다.

정상 FOLLOW, 정상 STOP, 중복 key, 상태 모순을 같은 parser에 통과시켜 승인과 거부를 비교합니다. 실제 이미지 결과는 원문과 화면을 함께 읽습니다.

다음 단원에서는 검증된 판단을 같은 프레임의 좌표와 연결해 실제 속도로 바꿉니다.

비교표와 실제 화면·로그를 강사 설명과 대조합니다. 별도 답안 작성은 없으며, 자동 저장 파일과 실행 결과로 진행 상태를 확인합니다.
